#  Модуль 8. Кодирование категориальных признаков

## Подробный конспект

### 8.1. Зачем переводить категории в числа

Модели машинного обучения — это математические алгоритмы. Они умеют складывать, умножать, считать расстояния и производные. Но они **не понимают слова** «Москва», «синий» или «премиум».

**Пример:** у вас есть признак «город» с значениями: Москва, СПб, Казань. Если просто заменить их на числа 1, 2, 3 (Москва=1, СПб=2, Казань=3), модель подумает, что **Казань в 3 раза больше Москвы**, а **СПб — это среднее между ними**. Это бессмыслица.

Задача кодирования — перевести категории в числа так, чтобы **сохранить информацию и не придумать ложных отношений** между категориями.

### 8.2. Типы категориальных признаков

Перед выбором метода кодирования нужно понять, с чем мы имеем дело.

#### Номинальные (Nominal)

Категории без порядка. Нельзя сказать, что одна «больше» другой.

- Город: Москва, СПб, Казань
- Цвет: красный, зелёный, синий
- Марка автомобиля: Toyota, BMW, Lada

#### Порядковые (Ordinal)

Категории с естественным порядком. Можно сравнивать.

- Образование: начальное < среднее < высшее < аспирантура
- Класс обслуживания: эконом < комфорт < бизнес < первый
- Уровень удовлетворённости: плохо < средне < хорошо < отлично

> **Важно:** для порядковых признаков можно использовать специальное кодирование (Ordinal Encoding), а для номинальных — нельзя просто назначить 1, 2, 3, потому что появится ложный порядок.

### 8.3. Label Encoding

**Идея:** каждой уникальной категории присваивается целое число: 0, 1, 2, 3...

In [1]:
from sklearn.preprocessing import LabelEncoder

города = ['Москва', 'СПб', 'Казань', 'Москва', 'СПб', 'Москва']

encoder = LabelEncoder()
коды = encoder.fit_transform(города)

print(коды)          # [1 2 0 1 2 1] — порядок зависит от алфавита
print(encoder.classes_)  # ['Казань' 'Москва' 'СПб']

[1 2 0 1 2 1]
['Казань' 'Москва' 'СПб']


**Обратное преобразование:**

In [2]:
original = encoder.inverse_transform([0, 1, 2])
print(original)  # ['Казань' 'Москва' 'СПб']

['Казань' 'Москва' 'СПб']


#### Проблема Label Encoding

Модели, основанные на расстояниях или линейных комбинациях, воспримут числа как **непрерывную шкалу**:

- Москва = 1, СПб = 2 -> модель думает, что СПб = Москва + 1
- Это создаёт ложные отношения: «СПб больше Москвы», «Казань меньше всех»

**Когда можно использовать Label Encoding:**
- Деревья решений, случайный лес, градиентный бустинг (XGBoost, LightGBM, CatBoost) — они не чувствительны к числовым значениям категорий
- Когда категорий очень много и другие методы дают слишком разреженные данные

**Когда НЕЛЬЗЯ использовать:**
- Линейная регрессия, логистическая регрессия, SVM, KNN, нейросети — они воспримут числа как порядок

### 8.4. Ordinal Encoding

Специальный случай для **порядковых** признаков. Мы **сами** задаём числа, отражающие порядок.

In [3]:
import pandas as pd

df = pd.DataFrame({'образование': ['среднее', 'высшее', 'начальное', 'высшее', 'аспирантура']})

# Задаём порядок явно
порядок = ['начальное', 'среднее', 'высшее', 'аспирантура']

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(categories=[порядок])
df['образование_код'] = encoder.fit_transform(df[['образование']])

print(df)
#      образование  образование_код
# 0       среднее              1.0
# 1        высшее              2.0
# 2     начальное              0.0
# 3        высшее              2.0
# 4    аспирантура              3.0

   образование  образование_код
0      среднее              1.0
1       высшее              2.0
2    начальное              0.0
3       высшее              2.0
4  аспирантура              3.0


**Ключевой момент:** мы сами определили, что `начальное < среднее < высшее < аспирантура`. Модель получит осмысленную шкалу.

**Когда использовать:**
- Шкалы Ликерта («совсем не согласен» — «полностью согласен»)
- Уровни образования, классы обслуживания, рейтинги

**Опасность:** если расстояния между категориями неравномерны («среднее» — это ровно посередине между «начальным» и «высшим»?), модель может ошибаться. Но для деревьев это не критично.

### 8.5. One-Hot Encoding (OHE)

Самый популярный метод для номинальных категорий. **Не придумывает порядка**, а создаёт отдельный столбец для каждой категории.

**Идея:**
- Было: один столбец «город» с тремя значениями
- Стало: три столбца — «город_Москва», «город_СПб», «город_Казань»
- В каждой строке только одна «1», остальные «0»

In [6]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

df = pd.DataFrame({'город': ['Москва', 'СПб', 'Казань', 'Москва']})

encoder = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' — см. ниже
encoded = encoder.fit_transform(df[['город']])

# Превращаем в DataFrame
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['город'])
)

encoded_df

,город_Москва,город_СПб
0,1.0,0.0
1,0.0,1.0
2,0.0,0.0
3,1.0,0.0


#### Dummy Trap: зачем `drop='first'`?

Если оставить все три столбца, возникает **мультиколлинеарность** (см. Модуль 4):

In [ ]:
город_Москва + город_СПб + город_Казань = 1 (всегда!)

Это означает, что один столбец **линейно выражается** через два других. Для линейных моделей это плохо — коэффициенты становятся нестабильными.

**Решение:** выбросить (drop) один столбец. Он становится «базовой категорией», и модель понимает: если в остальных нули — значит, это базовая категория.

In [7]:
# Без drop='first' — все категории
encoder_full = OneHotEncoder(sparse_output=False)
encoded_full = encoder_full.fit_transform(df[['город']])
print(encoded_full.shape)  # (4, 3)

# С drop='first' — на одну меньше
encoder_drop = OneHotEncoder(sparse_output=False, drop='first')
encoded_drop = encoder_drop.fit_transform(df[['город']])
print(encoded_drop.shape)  # (4, 2)

(4, 3)
(4, 2)


**Когда НЕ нужно drop='first':**
- Деревья и бустинги — им всё равно
- Регуляризация (Ridge, Lasso) часто справляется с коллинеарностью
- Нейросети

#### One-Hot через pandas (быстро для EDA)

In [8]:
# Быстрый способ без sklearn
pd.get_dummies(df, columns=['город'], drop_first=True)

,город_Москва,город_СПб
0,True,False
1,False,True
2,False,False
3,True,False


#### Проблема OHE: слишком много категорий

Если признак «ID пользователя» или «почтовый индекс» имеет 10 000 уникальных значений, OHE создаст 10 000 столбцов.

**Последствия:**
- Разреженная матрица (sparse) — почти все нули
- Проклятие размерности
- Переобучение (особенно на редких категориях)

**Решения:**
- Объединить редкие категории в «Другое» (см. раздел 8.8)
- Использовать Target Encoding или Hash Encoding
- Использовать встроенную обработку категорий в CatBoost / LightGBM

### 8.6. Target Encoding (Mean Encoding)

**Идея:** заменить категорию на **среднее значение целевой переменной** (target) для этой категории.

**Пример:** предсказываем вероятность дефолта по кредиту. Заменяем «город» на средний дефолт в этом городе.

In [9]:
import pandas as pd

df = pd.DataFrame({
    'город': ['Москва', 'СПб', 'Москва', 'Казань', 'СПб', 'Москва'],
    'дефолт': [0, 1, 0, 0, 1, 0]  # целевая переменная: 1 = дефолт
})

# Считаем средний дефолт по городам
средний_дефолт = df.groupby('город')['дефолт'].mean()
print(средний_дефолт)
# город
# Казань    0.000000
# Москва    0.000000
# СПб       1.000000

# Кодируем
df['город_encoded'] = df['город'].map(средний_дефолт)
df

город
Казань    0.0
Москва    0.0
СПб       1.0
Name: дефолт, dtype: float64


,город,дефолт,город_encoded
0,Москва,0,0.0
1,СПб,1,1.0
2,Москва,0,0.0
3,Казань,0,0.0
4,СПб,1,1.0
5,Москва,0,0.0


#### Опасность: переобучение (Data Leakage)

Если мы считаем средний таргет **по всем данным**, включая текущую строку, информация о таргете «утекает» в признак. Модель «подсмотрит» ответ.

**Решение 1: Smoothing (сглаживание)**

Смешиваем среднее категории с глобальным средним, особенно если наблюдений в категории мало.

$${encoded} = \frac{n \times {mean\_category} + m \times {mean\_global}}{n + m}$$

Где:
- $n$ — число наблюдений в категории
- ${mean\_category}$ — средний таргет в категории
- $m$ — параметр сглаживания (обычно 1, 10 или 300)
- ${mean\_global}$ — глобальное среднее таргета

In [11]:
global_mean = df['дефолт'].mean()
m = 1  # параметр сглаживания

def smoothed_mean(group):
    n = len(group)
    category_mean = group.mean()
    return (n * category_mean + m * global_mean) / (n + m)

df['город_smoothed'] = df['город'].map(df.groupby('город')['дефолт'].apply(smoothed_mean))
df

,город,дефолт,город_encoded,город_smoothed
0,Москва,0,0.0,0.083333
1,СПб,1,1.0,0.777778
2,Москва,0,0.0,0.083333
3,Казань,0,0.0,0.166667
4,СПб,1,1.0,0.777778
5,Москва,0,0.0,0.083333


**Решение 2: CV-Loop (кросс-валидационное кодирование)**

Для каждой строки среднее считается **без неё самой**, используя другие фолды.

In [14]:
from sklearn.model_selection import KFold

def target_encode_cv(df, column, target, n_splits=5, smoothing=1.0):
    df = df.copy()
    global_mean = df[target].mean()
    df[f'{column}_encoded'] = 0.0
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for train_idx, val_idx in kf.split(df):
        # Считаем среднее на трейне
        train_data = df.iloc[train_idx]
        means = train_data.groupby(column)[target].mean()
        
        # Применяем к валидации
        df.loc[val_idx, f'{column}_encoded'] = df.loc[val_idx, column].map(means)
    
    # Заполняем пропуски (если категория попалась только в одном фолде)
    df[f'{column}_encoded'] = df[f'{column}_encoded'].fillna(global_mean)
    
    return df

df_encoded = target_encode_cv(df, 'город', 'дефолт')
print(df_encoded)

    город  дефолт  город_encoded  город_smoothed
0  Москва       0       0.000000        0.083333
1     СПб       1       1.000000        0.777778
2  Москва       0       0.000000        0.083333
3  Казань       0       0.333333        0.166667
4     СПб       1       1.000000        0.777778
5  Москва       0       0.000000        0.083333


**Когда использовать Target Encoding:**
- Категорий много (OHE даст тысячи столбцов)
- Категории имеют **сильную связь** с таргетом
- Деревья, бустинги, нейросети

**Когда НЕ использовать:**
- Категорий мало — OHE лучше
- Таргет сильно разбалансирован — редкие категории шумят
- Без контроля переобучения — обязательно smoothing или CV

### 8.7. Frequency Encoding

**Идея:** заменить категорию на **частоту** её появления (или долю от всех наблюдений).

In [18]:
freq = df['город'].value_counts(normalize=True)
df['город_freq'] = df['город'].map(freq)

df[['город', 'город_freq']].drop_duplicates()
#      город  город_freq
# 0   Москва    0.500000
# 1      СПб    0.333333
# 3   Казань    0.166667

,город,город_freq
0,Москва,0.500000
1,СПб,0.333333
3,Казань,0.166667


**Смысл:** частые категории могут вести себя иначе, чем редкие. Например, популярный товар vs. нишевый.

**Дополнение:** можно считать не частоту, а **ранг** частоты (1 — самая частая, 2 — вторая и т.д.).

### 8.8. Работа с редкими категориями

Если категория встречается 1–2 раза в 100 000 строк:
- OHE создаст почти пустой столбец
- Target Encoding будет шумным (среднее по 1–2 точкам ненадёжно)
- Модель переобучится на этих редких случаях

**Решение: объединить в «Other»**

In [20]:
# Задаём порог
min_frequency = 2

# Считаем частоты
value_counts = df['город'].value_counts()

# Категории, которые встречаются реже порога
rare = value_counts[value_counts < min_frequency].index

# Заменяем
df['город_grouped'] = df['город'].replace(rare, 'Other')

df['город_grouped'].value_counts()

город_grouped
Москва    3
СПб       2
Other     1
Name: count, dtype: int64

**Альтернатива:** объединять не в «Other», а в более крупную группу по доменному знанию. Например, редкие марки автомобилей -> «Премиум» или «Китайские».

### 8.9. Binary Encoding

**Идея:** промежуточный метод между Label Encoding и OHE.

1. Сначала Label Encoding: Москва=1, СПб=2, Казань=3
2. Затем переводим число в **двоичный код**: 1=01, 2=10, 3=11
3. Каждый бит — отдельный столбец

**Зачем:** для 100 категорий OHE даст 100 столбцов, а Binary Encoding даст только $\log_2(100) \approx 7$ столбцов.

> Требуется установка: `pip install category_encoders`

In [30]:
import category_encoders as ce

encoder = ce.BinaryEncoder(cols=['город'])
df_binary = encoder.fit_transform(df[['город']])

pd.concat([df['город'], df_binary], axis=1)

,город,город_0,город_1
0,Москва,0,1
1,СПб,1,0
2,Москва,0,1
3,Казань,1,1
4,СПб,1,0
5,Москва,0,1


### 8.10. Hash Encoding

**Идея:** применить хеш-функцию к названию категории и получить число в заданном диапазоне.

- 10 000 категорий -> фиксированные 16 или 32 столбца
- Коллизии возможны (разные категории попадут в один бин), но на практике часто приемлемо

In [34]:
encoder = ce.HashingEncoder(cols=['город'], n_components=8)
df_hash = encoder.fit_transform(df[['город']])
pd.concat([df['город'], df_hash], axis=1)

,город,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7
0,Москва,0,0,0,0,0,0,1,0
1,СПб,0,0,0,0,1,0,0,0
2,Москва,0,0,0,0,0,0,1,0
3,Казань,0,0,0,0,0,0,0,1
4,СПб,0,0,0,0,1,0,0,0
5,Москва,0,0,0,0,0,0,1,0


**Когда использовать:** очень много категорий, скорость важнее точности (например, в продакшене с высоким трафиком).

### 8.11. Embeddings (упоминание)

Для нейросетей и сложных категорий (например, ID пользователей, тексты) используют **эмбеддинги** — плотные векторы низкой размерности, которые обучаются вместе с моделью.

**Пример:** в рекомендательных системах каждый пользователь и каждый товар — это вектор из 50–200 чисел. Похожие пользователи получают похожие векторы.

Это выходит за рамки базового курса, но важно знать, что такой метод существует.

### 8.12. Сравнение методов кодирования

| Метод | Тип категорий | Число столбцов | Подходит для линейных моделей | Подходит для деревьев | Риск переобучения | Комментарий |
|-------|--------------|----------------|------------------------------|----------------------|-------------------|-------------|
| **Label Encoding** | Любые | 1 | FALSE Нет | TRUE Да | Низкий | Создаёт ложный порядок |
| **Ordinal Encoding** | Порядковые | 1 | ️ Осторожно | TRUE Да | Низкий | Только когда порядок реален |
| **One-Hot Encoding** | Номинальные | N или N-1 | TRUE Да | TRUE Да | Низкий | Лучший для мало категорий |
| **Target Encoding** | Любые | 1 | TRUE Да | TRUE Да | **Высокий** | Требует CV / smoothing |
| **Frequency Encoding** | Любые | 1 | TRUE Да | TRUE Да | Низкий | Простой, но менее информативный |
| **Binary Encoding** | Номинальные | log₂(N) | TRUE Да | TRUE Да | Низкий | Компромисс размерности |
| **Hash Encoding** | Номинальные | Задано | TRUE Да | TRUE Да | Средний | Коллизии возможны |

### 8.13. Практический пример: полный цикл кодирования

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

# Создаём данные
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'город': np.random.choice(['Москва', 'СПб', 'Казань', 'Новосибирск', 'Екатеринбург'], n),
    'класс_обслуживания': np.random.choice(['эконом', 'комфорт', 'бизнес', 'первый'], n),
    'удовлетворенность': np.random.choice(['низкая', 'средняя', 'высокая', 'отличная'], n),
    'доход': np.random.lognormal(11, 0.5, n),
    'таргет': np.random.binomial(1, 0.3, n)  # бинарная цель
})

# === 1. ORDINAL ENCODING: удовлетворенность (естественный порядок) ===
порядок_удовл = [['низкая', 'средняя', 'высокая', 'отличная']]
ord_encoder = OrdinalEncoder(categories=порядок_удовл)
df['удовлетворенность_код'] = ord_encoder.fit_transform(df[['удовлетворенность']])

# === 2. ONE-HOT ENCODING: город (номинальный, категорий мало) ===
ohe = OneHotEncoder(sparse_output=False, drop='first')
город_ohe = ohe.fit_transform(df[['город']])
город_ohe_df = pd.DataFrame(город_ohe, columns=ohe.get_feature_names_out(['город']))
df = pd.concat([df, город_ohe_df], axis=1)

# === 3. TARGET ENCODING: класс обслуживания (связь с таргетом) ===
# Сглаживание + защита от переобучения: считаем на части данных
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)

# Считаем средний таргет на трейне
global_mean = train['таргет'].mean()
smoothing = 10

train_stats = train.groupby('класс_обслуживания').agg(
    n=('таргет', 'size'),
    mean=('таргет', 'mean')
).reset_index()

train_stats['encoded'] = (train_stats['n'] * train_stats['mean'] + smoothing * global_mean) / (train_stats['n'] + smoothing)

# Маппим на train и test
mapping = dict(zip(train_stats['класс_обслуживания'], train_stats['encoded']))
train['класс_target'] = train['класс_обслуживания'].map(mapping)
test['класс_target'] = test['класс_обслуживания'].map(mapping).fillna(global_mean)

# === 4. FREQUENCY ENCODING: город ===
freq_map = df['город'].value_counts(normalize=True).to_dict()
df['город_freq'] = df['город'].map(freq_map)

# === 5. РЕДКИЕ КАТЕГОРИИ (демонстрация) ===
# Добавим редкую категорию
df.loc[0:5, 'город'] = 'Редкий_Город'

# Объединяем редкие (менее 10 раз)
threshold = 10
counts = df['город'].value_counts()
rare = counts[counts < threshold].index
df['город_clean'] = df['город'].replace(rare, 'Other')

print("После обработки редких категорий:")
print(df['город_clean'].value_counts())

# Итоговый набор признаков для модели
feature_cols = ['доход', 'удовлетворенность_код', 'класс_target', 'город_freq'] + \
               list(город_ohe_df.columns)

print(f"\nИтоговые признаки: {feature_cols}")
print(df[feature_cols].head())

### 8.14. Чек-лист для самопроверки

Перед переходом к Модулю 9 убедитесь, что вы:

- [ ] Понимаете, почему нельзя просто заменить «Москва» на 1, «СПб» на 2
- [ ] Можете отличить номинальные категории от порядковых
- [ ] Знаете, когда можно использовать Label Encoding, а когда нельзя
- [ ] Умеете применять Ordinal Encoding с заданным порядком
- [ ] Можете выполнить One-Hot Encoding через `OneHotEncoder` и `pd.get_dummies`
- [ ] Понимаете, что такое Dummy Trap и зачем использовать `drop='first'`
- [ ] Знаете, в чём суть Target Encoding
- [ ] Понимаете риск переобучения при Target Encoding
- [ ] Умеете применять smoothing и CV-loop для защиты от переобучения
- [ ] Знаете, зачем объединять редкие категории в «Other»
- [ ] Понимаете, что такое Frequency Encoding и когда он полезен
- [ ] Знаете о существовании Binary Encoding, Hash Encoding и Embeddings
- [ ] Можете выбрать подходящий метод кодирования под конкретную ситуацию

> **Переход к Модулю 9:** Категории превращены в числа, но числовые признаки всё ещё имеют разные масштабы: возраст измеряется в годах, доход — в рублях, а удовлетворённость — в баллах. Следующий модуль научит приводить все числовые признаки к сопоставимым масштабам — масштабированию и нормализации.